# 9. Bias in the data: Titanic survival

BDI's `ebdai` package looks at **bias in the labels** and, after a fuzzy rule model is fitted, **bias in the rules that fire**. This notebook uses the Titanic table from the [WorkshopIgualdad2025](https://github.com/rferper/WorkshopIgualdad2025) workshop (Raquel Fernandez Peralta and Javier Fumanal Idocin), originally written for Ex-Fuzzy 2.1.3. The current `ex_fuzzy` API still trains the classifier; the new helpers live in `ebdai`.

`import ex_fuzzy` is the rule learner. `import ebdai` is the bias layer.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from ex_fuzzy import BaseFuzzyRulesClassifier, FUZZY_SETS
from ex_fuzzy import eval_tools

from ebdai import (
    features_and_target,
    load_titanic,
    outcome_rates_by_group,
    plot_outcome_rates,
    plot_winning_rules_by_group,
    fairness_report,
    parse_printed_rules,
    winning_rules_by_group,
)

frame, sensitive = load_titanic()
X, y = features_and_target(frame, 'Survived')
print(X.head())
print('rows', len(X), 'sensitive', sensitive)

## Positive outcome by sex

Women survived at a much higher rate than men. That is **bias in the data**: the label `Survived` is associated with `Sex` before any model is trained.

In [ ]:
rates = outcome_rates_by_group(y, X[sensitive], positive_label=1)
print(rates)
plot_outcome_rates(rates, title='Titanic survival rate by sex')

## A small fuzzy rule classifier

The search is kept short so the notebook stays a demo. Current `BaseFuzzyRulesClassifier` still accepts `nRules`, `nAnts`, `n_linguistic_variables`, `ds_mode` and `categorical_mask` as in 2.1.3; missing values are rejected, which is why the loader dropped incomplete rows.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)
clf = BaseFuzzyRulesClassifier(
    nRules=8, nAnts=3, fuzzy_type=FUZZY_SETS.t1,
    n_linguistic_variables=3, ds_mode=1, verbose=False,
    n_gen=6, pop_size=12, patience=3, random_state=42,
)
clf.fit(X_train, y_train)
report = eval_tools.eval_fuzzy_model(
    clf, X_train, y_train, X_test, y_test,
    plot_rules=False, print_rules=True, plot_partitions=False,
    return_rules=True, bootstrap_results_print=False,
)
print(report[:1500] if report else '(no rule text)')

## Bias in inference: which rules fire for whom

`explainable_predict` still returns winning-rule indexes. `winning_rules_by_group` counts them per sex.

In [ ]:
y_pred = clf.predict(X_test)
table, gaps = fairness_report(y_test, y_pred, X_test[sensitive])
print(table)
print(pd.Series(gaps))

texts = parse_printed_rules(report or '')
counts = winning_rules_by_group(clf, X_test, X_test[sensitive], rule_texts=texts)
print(counts.head())
plot_winning_rules_by_group(counts, title='Winning Titanic rules by sex')